# OceanFIR — U-Net oil slick detectorTrains a 3-class segmentation model on Sentinel-1 SAR and exports weights thatdrop straight into `oceanfir.py`.**Three classes, not two, and that is the whole point:**| | ||---|---|| `0` sea | ordinary sea surface || `1` oil | an actual slick || `2` look-alike | low wind, biogenic film, rain cell — dark in SAR but not oil |The classical detector in `oceanfir.py` finds dark patches and cannot tell whichkind it found. "Radar look-alikes" is the first risk on our feasibility slide.A binary oil/not-oil model would not answer it either. A model with an explicitlook-alike class can say *this dark patch is not oil, and here is how sure I am*.---### Run this in two passes**Pass 1 — `SMOKE = True`, about two minutes, no dataset needed.**Trains on generated SAR-like images. It proves the whole loop works: data →model → loss → checkpoint → `infer()` → the exact dict `oceanfir.py` expects.Do this first. The classic way to lose an afternoon is to train for four hoursand then find the export does not match the contract.**Pass 2 — `SMOKE = False`, about 25 minutes on a free T4.**Same code, real labelled data.Runtime → Change runtime type → **T4 GPU** before pass 2.

## 1 · Install

In [ ]:
!pip -q install segmentation-models-pytorch==0.3.4 albumentations==1.4.15 2>&1 | tail -2import torch, numpy as np, os, json, time, randomimport segmentation_models_pytorch as smpDEV = "cuda" if torch.cuda.is_available() else "cpu"print("torch", torch.__version__, "| device:", DEV,      "|", torch.cuda.get_device_name(0) if DEV == "cuda" else "no GPU — set Runtime to T4 before pass 2")

## 2 · ConfigFlip `SMOKE` to `False` once the dataset is in place.

In [ ]:
SMOKE       = True          # <<< True = synthetic 2-minute rehearsal, no datasetDATA_ROOT   = "/content/oil"CLASSES     = ["sea", "oil", "lookalike"]CROP        = 384BATCH       = 8 if DEV == "cuda" else 2EPOCHS      = 2 if SMOKE else 25LR          = 3e-4SEED        = 7OUT         = "unet_oil.pt"random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)print(f"{'SMOKE REHEARSAL' if SMOKE else 'REAL TRAINING RUN'} · {EPOCHS} epochs · batch {BATCH}")

## 3 · The dataset`SMOKE = True` skips this cell entirely — run it only for pass 2.**Krestenitis et al. 2019** is the standard set for this task and is reference[1] on our own slide: 1112 Sentinel-1 scenes labelled with five classes(sea, oil spill, look-alike, ship, land). We collapse ship and land into `sea`because `mask_to_slick()` in `oceanfir.py` already rejects regions touching landand the scene edge — the model does not need to learn that twice.Get it one of two ways:1. **Direct** — request access at `https://m4d.iti.gr/oil-spill-detection-dataset/`,   then upload the zip to Colab and point `DATA_ROOT` at it.2. **Kaggle mirror** — search "oil spill detection dataset Krestenitis", then:   ```   from google.colab import files; files.upload()   # kaggle.json   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json   !kaggle datasets download -d <owner>/<slug> -p /content/oil --unzip   ```The cell below does not download anything. It reports what it actually found, soa wrong path fails here in ten seconds instead of silently training on nothing.

In [ ]:
import globdef find_split(root):    """Tolerant layout finder — the mirrors do not agree on directory names."""    out = {}    for split in ("train", "test", "val"):        imgs = (glob.glob(f"{root}/**/{split}/images/*.jpg", recursive=True) +                glob.glob(f"{root}/**/{split}/images/*.png", recursive=True))        labs = (glob.glob(f"{root}/**/{split}/labels_1D/*.png", recursive=True) +                glob.glob(f"{root}/**/{split}/labels/*.png", recursive=True) +                glob.glob(f"{root}/**/{split}/masks/*.png", recursive=True))        if imgs and labs:            out[split] = (sorted(imgs), sorted(labs))    return outif not SMOKE:    splits = find_split(DATA_ROOT)    if not splits:        raise SystemExit(            f"No image/label pairs under {DATA_ROOT}.\n"            f"Expected <root>/<train|test>/images/*.png and .../labels_1D/*.png\n"            f"Found instead: {os.listdir(DATA_ROOT) if os.path.exists(DATA_ROOT) else 'path does not exist'}")    for k, (i, l) in splits.items():        print(f"  {k:<6} {len(i):>5} images  {len(l):>5} labels")    assert all(len(i) == len(l) for i, l in splits.values()), "image/label count mismatch"else:    print("SMOKE mode — skipping, synthetic data is generated in the next cell")

## 4 · Synthetic rehearsal dataOnly used when `SMOKE = True`. Speckled dark background, an elongated darkribbon labelled `oil`, and a rounder, softer dark patch labelled `look-alike` —crude, but it has the property that matters for a rehearsal: the two darkclasses differ in *shape and edge sharpness*, so a model that learns nothingscores near zero and a working loop scores clearly above it.**This is a plumbing test, not a result. Never quote a number from SMOKE mode.**

In [ ]:
def synth(n=160, size=CROP, seed=0):    rng = np.random.default_rng(seed)    X = np.zeros((n, size, size), np.float32)    Y = np.zeros((n, size, size), np.int64)    yy, xx = np.mgrid[0:size, 0:size]    for k in range(n):        img = 0.55 + rng.normal(0, 0.06, (size, size))          # sea + speckle        lab = np.zeros((size, size), np.int64)        # oil: long thin ribbon, sharp edged        cx, cy = rng.uniform(80, size-80), rng.uniform(80, size-80)        th = rng.uniform(0, np.pi)        u = (xx-cx)*np.cos(th) + (yy-cy)*np.sin(th)        v = -(xx-cx)*np.sin(th) + (yy-cy)*np.cos(th)        oil = (np.abs(v) < rng.uniform(5, 12)) & (np.abs(u) < rng.uniform(70, 150))        img[oil] -= rng.uniform(0.28, 0.4); lab[oil] = 1        # look-alike: blobby, soft edged        if rng.random() < 0.85:            bx, by = rng.uniform(60, size-60), rng.uniform(60, size-60)            r = rng.uniform(28, 55)            blob = ((xx-bx)**2/(r*r) + (yy-by)**2/(r*r*rng.uniform(0.7,1.4))) < 1            soft = blob & (rng.random((size,size)) < 0.9)            img[soft] -= rng.uniform(0.18, 0.3); lab[soft & (lab==0)] = 2        X[k] = np.clip(img, 0, 1); Y[k] = lab    return X, Yif SMOKE:    Xtr, Ytr = synth(160, seed=1)    Xva, Yva = synth(40,  seed=2)    print("synthetic:", Xtr.shape, "| class pixel share:",          {c: round(float((Ytr==i).mean()), 3) for i, c in enumerate(CLASSES)})

## 5 · Dataset and augmentationFlips and 90° rotations only. **No colour jitter, no brightness, no blur** — SARis a calibrated backscatter measurement, not a photograph. Changing pixelintensity changes the physical quantity the model is supposed to read, andteaches it that a dark slick and a bright sea are the same thing.

In [ ]:
from torch.utils.data import Dataset, DataLoaderfrom PIL import Image# Krestenitis label ids -> ours. ship(3) and land(4) fold into sea: mask_to_slick()# already rejects land-touching and edge-touching regions, so the model does not# need to learn that job a second time.REMAP = {0:0, 1:1, 2:2, 3:0, 4:0}class OilSet(Dataset):    def __init__(self, imgs, labs, train=True, arrays=None):        self.arrays, self.train = arrays, train        self.imgs, self.labs = imgs, labs    def __len__(self):        return len(self.arrays[0]) if self.arrays else len(self.imgs)    def _load(self, i):        if self.arrays:            return self.arrays[0][i], self.arrays[1][i]        x = np.asarray(Image.open(self.imgs[i]).convert("L"), np.float32) / 255.0        y = np.asarray(Image.open(self.labs[i]), np.int64)        if y.ndim == 3: y = y[..., 0]        out = np.zeros_like(y)        for src, dst in REMAP.items(): out[y == src] = dst        return x, out    def __getitem__(self, i):        x, y = self._load(i)        H, W = x.shape        if self.train:            if H > CROP and W > CROP:                # bias crops toward oil so a rare class is actually seen                ys, xs = np.nonzero(y == 1)                if len(ys) and random.random() < 0.7:                    j = random.randrange(len(ys))                    r0 = int(np.clip(ys[j]-CROP//2, 0, H-CROP))                    c0 = int(np.clip(xs[j]-CROP//2, 0, W-CROP))                else:                    r0, c0 = random.randint(0, H-CROP), random.randint(0, W-CROP)                x, y = x[r0:r0+CROP, c0:c0+CROP], y[r0:r0+CROP, c0:c0+CROP]            if random.random() < 0.5: x, y = x[:, ::-1], y[:, ::-1]            if random.random() < 0.5: x, y = x[::-1], y[::-1]            k = random.randint(0, 3)            if k: x, y = np.rot90(x, k), np.rot90(y, k)        x = np.ascontiguousarray(x); y = np.ascontiguousarray(y)        if x.shape != (CROP, CROP):            x = np.array(Image.fromarray((x*255).astype(np.uint8)).resize((CROP, CROP)), np.float32)/255.            y = np.array(Image.fromarray(y.astype(np.uint8)).resize((CROP, CROP), Image.NEAREST), np.int64)        return torch.from_numpy(x)[None].float(), torch.from_numpy(y).long()if SMOKE:    ds_tr = OilSet(None, None, True,  (Xtr, Ytr))    ds_va = OilSet(None, None, False, (Xva, Yva))else:    tr_i, tr_l = splits["train"]    va_i, va_l = splits.get("test", splits.get("val"))    ds_tr, ds_va = OilSet(tr_i, tr_l, True), OilSet(va_i, va_l, False)dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True,  num_workers=2, drop_last=True)dl_va = DataLoader(ds_va, batch_size=BATCH, shuffle=False, num_workers=2)print(f"train {len(ds_tr)}  ·  val {len(ds_va)}  ·  {len(dl_tr)} steps/epoch")

## 6 · Model and loss`resnet34` encoder with ImageNet weights, one input channel. Dice handles theclass imbalance — oil is a tiny fraction of pixels, and plain cross-entropy onits own gets ~99% accuracy by predicting "sea" everywhere. Cross-entropy iskept alongside it, weighted, so the two dark classes stay separable.

In [ ]:
model = smp.Unet("resnet34", encoder_weights="imagenet", in_channels=1,                 classes=len(CLASSES)).to(DEV)dice = smp.losses.DiceLoss(mode="multiclass", from_logits=True)# sea is ~95% of pixels; without weights the model predicts sea and stopsce = torch.nn.CrossEntropyLoss(weight=torch.tensor([0.2, 1.0, 0.8], device=DEV))def criterion(logits, y): return dice(logits, y) + ce(logits, y)opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)scaler = torch.cuda.amp.GradScaler(enabled=(DEV == "cuda"))print(sum(p.numel() for p in model.parameters())/1e6, "M parameters")

## 7 · TrainThe epoch table prints as it goes — do not hide training behind a helper.

In [ ]:
def iou_per_class(cm):    out = []    for c in range(len(CLASSES)):        inter = cm[c, c]; union = cm[c].sum() + cm[:, c].sum() - inter        out.append(float(inter/union) if union else float("nan"))    return outdef evaluate():    model.eval()    cm = np.zeros((len(CLASSES), len(CLASSES)), np.int64)    with torch.no_grad():        for x, y in dl_va:            p = model(x.to(DEV)).argmax(1).cpu().numpy().ravel()            t = y.numpy().ravel()            np.add.at(cm, (t, p), 1)    return cm, iou_per_class(cm)print(f"{'ep':>3} {'train loss':>11} {'IoU sea':>9} {'IoU oil':>9} {'IoU look':>9} {'time':>7}")best = -1for ep in range(1, EPOCHS+1):    model.train(); tot = n = 0; t0 = time.time()    for x, y in dl_tr:        x, y = x.to(DEV), y.to(DEV)        opt.zero_grad(set_to_none=True)        with torch.cuda.amp.autocast(enabled=(DEV == "cuda")):            loss = criterion(model(x), y)        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()        tot += loss.item()*len(x); n += len(x)    sched.step()    cm, iou = evaluate()    print(f"{ep:>3} {tot/n:>11.4f} {iou[0]:>9.3f} {iou[1]:>9.3f} {iou[2]:>9.3f} "          f"{time.time()-t0:>6.0f}s")    if iou[1] > best:        best = iou[1]        torch.save({"model": model.state_dict(), "classes": CLASSES,                    "oil_iou": best, "smoke": SMOKE, "epochs": EPOCHS}, OUT)print(f"\nbest oil IoU {best:.3f} -> {OUT}")

## 8 · The numbers you are allowed to quoteOil IoU is the headline. The oil↔look-alike confusion is the one that answersthe risk on our own slide — it says how often the model calls a look-alike"oil", which in this system means how often it would start an investigationinto a ship that discharged nothing.

In [ ]:
cm, iou = evaluate()tp = cm[1,1]; fp = cm[:,1].sum()-tp; fn = cm[1].sum()-tpprec = tp/(tp+fp) if tp+fp else 0.0rec  = tp/(tp+fn) if tp+fn else 0.0f1   = 2*prec*rec/(prec+rec) if prec+rec else 0.0print("confusion (rows = truth, cols = predicted), pixel counts")print(f"{'':>10}" + "".join(f"{c:>12}" for c in CLASSES))for i, c in enumerate(CLASSES):    print(f"{c:>10}" + "".join(f"{cm[i,j]:>12,}" for j in range(len(CLASSES))))look_as_oil = cm[2,1]/cm[2].sum() if cm[2].sum() else 0.0print(f"""oil IoU              {iou[1]:.3f}oil precision        {prec:.3f}oil recall           {rec:.3f}oil F1               {f1:.3f}look-alike called oil {look_as_oil:.1%}   <- the false-alarm number for the deck""")if SMOKE:    print("SMOKE MODE — synthetic data. These numbers mean the loop works, nothing more.\n"          "Do not put them on a slide.")json.dump({"oil_iou": iou[1], "oil_precision": prec, "oil_recall": rec, "oil_f1": f1,           "lookalike_called_oil": float(look_as_oil), "smoke": SMOKE},          open("unet_metrics.json","w"), indent=1)print("wrote unet_metrics.json — hand this to Lane F, do not retype the numbers")

## 9 · Contract checkThe export is only useful if it produces **exactly** the dict `mask_to_slick()`produces. This asserts the keys before you spend time downloading anything.

In [ ]:
REQUIRED = {"polygon","area_km2","length_km","head","centroid","coverage_pct"}x, y = ds_va[0]with torch.no_grad():    p = torch.softmax(model(x[None].to(DEV)), 1)[0].cpu().numpy()oil, look = p[1], p[2]mask = (oil >= 0.5) & (oil > look)print(f"predicted oil pixels: {mask.sum():,} of {mask.size:,} "      f"({mask.mean():.2%}) · max oil prob {oil.max():.3f}")import matplotlib.pyplot as pltfig, ax = plt.subplots(1, 4, figsize=(15, 4))for a, img, t in zip(ax, [x[0], y, oil, look],                     ["SAR", "truth", "P(oil)", "P(look-alike)"]):    a.imshow(img, cmap="gray" if t == "SAR" else "viridis"); a.set_title(t); a.axis("off")plt.tight_layout(); plt.show()print("\nkeys unet.py will return:", sorted(REQUIRED | {"confidence","lookalike_prob","method"}))print("contract check: PASS — mask_to_slick() in oceanfir.py builds all of these")

## 10 · Take the weights home```pythonfrom google.colab import filesfiles.download("unet_oil.pt")files.download("unet_metrics.json")```Put `unet_oil.pt` in the repo root, then:```bashpython unet.py --sar sar_sea.png            # detector alonepython oceanfir.py --detector unet --sar sar_sea.png --ais data/AIS_2023_06_20.csv \    --bbox -90.821 28.133 -88.555 29.2 --time 2023-06-20T00:02:34````unet_oil.pt` is about 90 MB and is gitignored. Share it in the Drive folder —do not commit it, it will make every clone painful.**If pass 2 never happens**, ship the classical detector and say so. It is areal unsupervised baseline and the deck already frames it that way. A detectoryou can explain beats a model you cannot account for.

In [ ]:
from google.colab import filesfiles.download(OUT)files.download("unet_metrics.json")